In [16]:
#load data zwn
import pandas as pd
raw_zwn = pd.read_csv("../data/raw/zwn_meldungen.csv")

#Clean data
#select useful columns
processed_zwn = raw_zwn[["service_name","requested_datetime","e","n","status","updated_datetime"]]


#define new columns names
new_names = {
"service_name":"category",
"e":"East",
"n":"North",
"requested_datetime":"report_time",
"updated_datetime":"resolved_time",
    }
processed_zwn= processed_zwn.rename(columns=new_names)

# convert datatype of "report_time" to datetime64
processed_zwn["report_time"] = pd.to_datetime(processed_zwn["report_time"], format ="%Y-%m-%dT%H:%M:%S")
processed_zwn["resolved_time"] = pd.to_datetime(processed_zwn["resolved_time"], format ="%Y-%m-%dT%H:%M:%S")

# create geometry category and geodataframe
import geopandas as gpd
from shapely.geometry import Point
processed_zwn = gpd.GeoDataFrame(processed_zwn,geometry=gpd.points_from_xy(processed_zwn["East"], processed_zwn["North"])
)
# select useful columns
processed_zwn= processed_zwn[["category","report_time","geometry","status","resolved_time"]]


# check missing values
#missing_count = processed_zwn[["category","report_time","geometry"]].isna().sum()
#print(f"The table shows the missing values in the dataframe {missing_count}")

#define missing CRS (CH1903+ / LV95)
processed_zwn = processed_zwn.set_crs(epsg=2056)

# load data quartiere
raw_quartiere = pd.read_csv("../data/raw/quartiere_zürich.csv")
processed_quartiere = raw_quartiere[["qname","geometry"]]
processed_quartiere.head(3)

#define new columns names
new_names1 = {
"qname":"Quartier",
    "geometry": "Geometry"
}
processed_quartiere= processed_quartiere.rename(columns=new_names1)

# check missing values
#missing_count = processed_quartiere[["Quartier","Geometry"]].isna().sum()
#print(f"The table shows the missing values in the dataframe {missing_count}")

#transform geometry datatype
#wkt.loads transform string datatype to geometry data type
from shapely import wkt
processed_quartiere["Geometry"] = processed_quartiere["Geometry"].apply(wkt.loads)

# create geodataframe to interprate "Geometry" as a geometry column
processed_quartiere = gpd.GeoDataFrame(
    processed_quartiere,
    geometry="Geometry")

#define missing CRS (CH1903+ / LV95)
processed_quartiere = processed_quartiere.set_crs(epsg=2056)

#download processed csv-files
#processed_zwn.to_file("../data/processed/processed_zwn.gpkg", driver="GPKG")
#processed_quartiere.to_file("../data/processed/processed_quartiere.gpkg", driver="GPKG")

# Question 1: "Wie unterscheidet sich die Bearbeitungszeit der Quartiere pro Kategorie ? "

#perform spatial join
zwn_with_quartiere = gpd.sjoin(processed_zwn,processed_quartiere, how="left", predicate ="within")

#filter data
#filter only reports which are already fixed
zwn_with_quartiere_filtred= zwn_with_quartiere[zwn_with_quartiere["status"] =="fixed - council"]
zwn_with_quartiere_category= zwn_with_quartiere_filtred

,category,report_time,geometry,status,resolved_time,index_right,Quartier
0,Strasse/Trottoir/Platz,2013-03-14 15:16:15,POINT (2678968 1247548),fixed - council,2013-04-12 07:59:30,16.0,Albisrieden
1,Strasse/Trottoir/Platz,2013-03-14 15:17:57,POINT (2680746 1249916),fixed - council,2013-04-12 08:00:22,20.0,Höngg
2,Strasse/Trottoir/Platz,2013-03-15 09:14:16,POINT (2684605 1251431),fixed - council,2013-04-12 08:08:10,33.0,Saatlen
3,Strasse/Trottoir/Platz,2013-03-15 09:17:15,POINT (2681754 1250376),fixed - council,2013-04-12 08:09:05,21.0,Wipkingen
4,Abfall/Sammelstelle,2013-03-15 10:36:53,POINT (2683094 1247762),fixed - council,2013-04-23 13:50:33,15.0,City
...,...,...,...,...,...,...,...
72403,Signalisation/Lichtsignal,2026-04-23 11:01:02,POINT (2682556 1247954),fixed - council,2026-04-23 14:06:31,2.0,Langstrasse
72404,Abfall/Sammelstelle,2026-04-23 11:33:16,POINT (2681319 1247393),fixed - council,2026-04-23 14:23:35,17.0,Sihlfeld
72405,Abfall/Sammelstelle,2026-04-23 12:53:51,POINT (2683470 1248995),fixed - council,2026-04-23 12:58:27,10.0,Oberstrass
72406,Signalisation/Lichtsignal,2026-04-23 13:41:53,POINT (2681816 1247528),fixed - council,2026-04-23 16:09:30,27.0,Werd
